# Séance 5 — Mini-cas de terrain

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/ts/notebooks/TS_S5_mini_cas.ipynb)
Enchaîner tout le travail sur le fichier brut, puis rédiger le rapport (avec un assistant IA, **vérifié**).

> Fichiers nécessaires : téléverser les CSV du dossier `TS_TP_donnees` (icône dossier à gauche dans Colab).
> Exécuter chaque cellule avec ▶ (ou Maj + Entrée).

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['sondages_miniers_terrain.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/ts/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
df = pd.read_csv('sondages_miniers_terrain.csv')

## Étape 1 — Contrôle qualité (reprendre la séance 2)

In [ ]:
df = df.drop_duplicates()
df['lithologie'] = df['lithologie'].str.strip().str.capitalize().replace({'Veine quartz': 'Veine de quartz'})
df.loc[df['au_gpt'] < 0, 'au_gpt'] = np.nan
m = df['au_gpt'] > 50; df.loc[m, 'au_gpt'] = df.loc[m, 'au_gpt'] / 1000
inv = df['de_m'] > df['a_m']; df.loc[inv, ['de_m', 'a_m']] = df.loc[inv, ['a_m', 'de_m']].values
x_max = 2000                      # À COMPLÉTER : limite de la zone du permis (m)
df = df[df['x_m'] <= x_max]       # retire le sondage mal positionné (à signaler !)
df = df.dropna(subset=['au_gpt'])
print(len(df), 'intervalles conservés')

## Étape 2 — Graphiques

In [ ]:
t = df.groupby('sondage').agg(x=('x_m', 'first'), y=('y_m', 'first'), au=('au_gpt', 'mean'))
plt.scatter(t['x'], t['y'], c=np.log10(t['au']), cmap='YlOrRd', s=80); plt.colorbar(); plt.title('Teneur moyenne par sondage'); plt.show()

## Étape 3 — Modèle

In [ ]:
df['quartz'] = (df['lithologie'] == 'Veine de quartz').astype(int)
df['mineralise'] = (df['au_gpt'] > 0.5).astype(int)
X = df[['quartz', 'sulfures_pct', 'alteration', 'de_m']]; y = df['mineralise']
Xa, Xt, ya, yt = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
arbre = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xa, ya)
print(export_text(arbre, feature_names=list(X.columns)))
print(confusion_matrix(yt, arbre.predict(Xt)))

## Étape 4 — Rapport
1. Copier **vos chiffres** (pas les données !) dans l’assistant IA avec une demande structurée : *Rôle, Situation, Demande, Limites, Forme*.
2. Vérifier : chaque chiffre du rapport existe-t-il dans ce notebook ? Rien n’est-il inventé ?
3. Ajouter la section **Limites** vous-même.